In [78]:
import numpy as np
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from utils.load import load_all_images, extract_load_features, log_confusion_matrix
from layers.Conv import Conv
from layers.NN import NeuralNetwork
from utils.logger import Logger

In [79]:
logger = Logger(log_dir='logs')

IMAGE_SIZE = (128, 128)
POOL_SIZE = 2
HIDDEN_LAYERS = [256, 128]
LEARNING_RATE = 0.01
EPOCHS = 100
DATA_ROOT = 'data/raw'

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

CLASS_TO_INDEX

INFO - Logger initialized. Log file: logs/training_log_20260330_210924.log


{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

In [80]:
logger.log_hyperparameters({
        'image_size': IMAGE_SIZE,
        'pool_size': POOL_SIZE,
        'hidden_layers': HIDDEN_LAYERS,
        'learning_rate': LEARNING_RATE,
        'epochs': EPOCHS,
})

INFO - ==================================================
INFO - HYPERPARAMETERS
INFO - ==================================================
INFO - image_size: (128, 128)
INFO - pool_size: 2
INFO - hidden_layers: [256, 128]
INFO - learning_rate: 0.01
INFO - epochs: 100
INFO - ==================================================


In [81]:
logger.info("[1/4] Memuat dataset...")
images_per_class = load_all_images(
   data_root=DATA_ROOT,
   class_names=CLASS_NAMES,
   image_size=IMAGE_SIZE,
)



INFO - [1/4] Memuat dataset...


  Loaded   400 gambar dari folder 'glioma'
  Loaded   400 gambar dari folder 'meningioma'
  Loaded   400 gambar dari folder 'notumor'
  Loaded   400 gambar dari folder 'pituitary'


In [82]:
len(images_per_class["glioma"])

400

In [83]:
all_images = []
all_labels = []
for class_name in CLASS_NAMES:
   imgs = images_per_class[class_name]
   all_images.append(imgs)
   all_labels.extend([CLASS_TO_INDEX[class_name]] * len(imgs))

In [84]:
logger.info(f"Jumlah Image {len(all_images) } dan Jumlah Label {len(all_labels)}")

INFO - Jumlah Image 4 dan Jumlah Label 1600


In [85]:
X_all = np.concatenate(all_images, axis=0)
y_all_labels = np.array(all_labels)



num_classes = len(CLASS_NAMES)
y_all = np.zeros((len(y_all_labels), num_classes))
print(y_all)

for i, label in enumerate(y_all_labels):
   y_all[i, label] = 1.0

print(y_all)

y_all_labels


[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 ...
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 ...
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]]


array([0, 0, 0, ..., 3, 3, 3], shape=(1600,))

In [86]:
X_train, X_test, y_train, y_test, y_train_labels, y_test_labels = train_test_split(
   X_all, y_all, y_all_labels,
   test_size=0.2,
   stratify=y_all_labels,
)

logger.info(f"Total data: {X_all.shape[0]} samples")
logger.info(f"Train/Test split: {X_train.shape[0]} train, {X_test.shape[0]} test (test_size=0.2, stratified)")

logger.log_dataset_info({
        'total_samples': X_all.shape[0],
        'train_samples': X_train.shape[0],
        'test_samples': X_test.shape[0],
        'image_size': IMAGE_SIZE,
        'num_classes': len(CLASS_NAMES),
        'classes': str(CLASS_NAMES),
    })


INFO - Total data: 1600 samples
INFO - Train/Test split: 1280 train, 320 test (test_size=0.2, stratified)
INFO - ==================================================
INFO - DATASET INFORMATION
INFO - ==================================================
INFO - total_samples: 1600
INFO - train_samples: 1280
INFO - test_samples: 320
INFO - image_size: (128, 128)
INFO - num_classes: 4
INFO - classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
INFO - ==================================================


In [87]:
logger.info("[2/4] Feature extraction Convolution...")
conv = Conv(pool_size=POOL_SIZE)
conv.info(logger=logger)


t0 = time.time()
X_train_feat, X_test_feat = extract_load_features(conv, X_train, X_test, logger)
logger.info(f"Waktu feature extraction: {time.time() - t0:.1f}s")
logger.info(f"Train features: {X_train_feat.shape}")
logger.info(f"Test  features: {X_test_feat.shape}")

INFO - [2/4] Feature extraction Convolution...
INFO - Conv Layer:
INFO -   Kernel size : 3x3
INFO -   Kernel      : [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]
INFO -   Pool size   : 2
INFO - Mengekstrak fitur training...
Training features: 100%|██████████| 1280/1280 [00:50<00:00, 25.12it/s]
INFO - Mengekstrak fitur testing...
Testing features: 100%|██████████| 320/320 [00:11<00:00, 27.70it/s]
INFO - Fitur disimpan ke data/extract/numpy/
INFO - Gambar hasil feature extraction disimpan ke data/extract/gambar/
INFO - Waktu feature extraction: 62.7s
INFO - Train features: (1280, 3969)
INFO - Test  features: (320, 3969)


In [88]:
feature_dim = conv.get_feature_dim(IMAGE_SIZE)
logger.info(f"Feature dimension: {feature_dim}")

INFO - Feature dimension: 3969


In [89]:
nn = NeuralNetwork(
   input_dim=feature_dim,
   hidden_layers=HIDDEN_LAYERS,
   output_dim=num_classes,
   learning_rate=LEARNING_RATE
)

In [90]:
nn.info(logger=logger)

t1 = time.time()
nn.train(
   X_train_feat,
   y_train,
   epochs=EPOCHS,
   logger=logger,
)
logger.info(f"Waktu training: {time.time() - t1:.1f}s")

INFO - Neural Network Architecture:
INFO -   Layers        : [3969, 256, 128, 4]
INFO -   Learning rate : 0.01
INFO -   Layer 1:  3969 ->  256  (ReLU)  [1,016,320 params]
INFO -   Layer 2:   256 ->  128  (ReLU)  [32,896 params]
INFO -   Layer 3:   128 ->    4  (Softmax)  [516 params]
INFO -   Total parameters: 1,049,732
Training: 100%|██████████| 100/100 [00:05<00:00, 19.94epoch/s, loss=0.6483, acc=0.8086]
INFO - Waktu training: 5.0s


In [91]:
logger.info("[4/4] Evaluasi Model...")
test_loss, test_acc = nn.evaluate(X_test_feat, y_test, y_labels=y_test_labels)
y_pred = nn.predict(X_test_feat)

logger.log_metrics({
   'test_loss': test_loss,
   'test_accuracy': test_acc,
})


INFO - [4/4] Evaluasi Model...
INFO - ==================================================
INFO - EVALUATION METRICS
INFO - ==================================================
INFO - test_loss: 0.8600
INFO - test_accuracy: 0.6500
INFO - ==================================================


In [92]:

logger.info("\nClassification Report:")
logger.info(classification_report(y_test_labels, y_pred, target_names=CLASS_NAMES))



INFO - 
Classification Report:
INFO -               precision    recall  f1-score   support

      glioma       0.57      0.55      0.56        80
  meningioma       0.53      0.40      0.46        80
     notumor       0.76      0.81      0.78        80
   pituitary       0.69      0.84      0.76        80

    accuracy                           0.65       320
   macro avg       0.64      0.65      0.64       320
weighted avg       0.64      0.65      0.64       320



In [93]:
# logger.info("\nConfusion Matrix:")
# log_confusion_matrix(y_test_labels, y_pred, CLASS_NAMES, logger)
# logger.info("✓ Project selesai!")